# 04 · Líneas base

**Fase CRISP-DM:** modelado (punto de partida).

**Objetivo:** medir cuánto se equivocan los métodos sin aprendizaje, por segmento, con el mismo protocolo que tendrán los modelos: walk-forward con horizonte 2 (con datos hasta M−1 se predice M+1). El estacional ingenuo, el mismo mes del año anterior, es el método actual de Lojagas y la vara que un modelo tiene que superar.

**Regla de privacidad:** solo usa la serie agregada por mes y segmento.

In [ ]:
import exp_a
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from lojagas.ml import metricas

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
exp_a.estilo_graficos()

H = exp_a.HORIZONTE
serie = exp_a.cargar_serie()
print(f"variables sin fuga temporal: {exp_a.verificar_sin_fuga(serie)} meses comprobados")
datos = exp_a.tabla_modelado(serie)
SEGMENTOS = sorted(datos["segmento"].unique())
print(
    f"meses evaluables: {datos['mes'].nunique()} "
    f"(de {datos['mes'].min():%Y-%m} a {datos['mes'].max():%Y-%m})"
)

## 1. Error por segmento

Cuatro referencias, ninguna con aprendizaje: el ingenuo (el último mes cerrado, t−2), el estacional ingenuo (t−12), el estacional ingenuo ajustado por los días de venta del mes (t−12 multiplicado por el cociente de días de venta entre este año y el anterior) y la media de los últimos 12 meses cerrados. TOTAL es el error sobre la suma de los segmentos. El sesgo es positivo cuando el método predice de más.

In [ ]:
ref = exp_a.evaluar_referencias(datos)
pruebas = ref["mes"].drop_duplicates()
print(f"meses de prueba: {len(pruebas)} (de {pruebas.min():%Y-%m} a {pruebas.max():%Y-%m})")

ORDEN = ["ingenuo", "estacional_ingenuo", "estacional_dias", "media_12"]
res = exp_a.resumir(ref)
print("WAPE (%)")
display(res.pivot_table(index="modelo", columns="segmento", values="wape").loc[ORDEN].round(2))
print("Todas las métricas")
columnas = ["wape", "mape", "mae", "rmse", "r2", "sesgo_pct", "n"]
display(res.set_index(["modelo", "segmento"])[columnas].loc[ORDEN].round(2))

## 2. ¿Depende el resultado del período de prueba?

Las referencias no se entrenan, así que también se pueden medir sobre todos los meses evaluables. Y los mismos errores se pueden recortar a las pruebas que tendría una ventana mínima más larga. Si el error cambia mucho entre períodos, la comparación con los modelos tiene que hacerse siempre sobre los mismos meses.

In [ ]:
COLUMNA = {
    "ingenuo": f"lag_{H}",
    "estacional_ingenuo": "lag_12",
    "estacional_dias": "estacional_dias",
    "media_12": "media_12",
}
filas = []
for nombre, columna in COLUMNA.items():
    for segmento, g in datos.groupby("segmento"):
        filas.append(
            {
                "periodo": f"todos los evaluables ({datos['mes'].nunique()} meses)",
                "modelo": nombre,
                "segmento": segmento,
                "wape": metricas.wape(g["toneladas"], g[columna]),
            }
        )
for ventana in (12, 18, 24):
    recorte = ref[ref["n_entrenamiento"] >= ventana]
    parcial = exp_a.resumir(recorte).query("segmento != 'TOTAL'")
    etiqueta = f"ventana {ventana} ({recorte['mes'].nunique()} meses)"
    filas += parcial.assign(periodo=etiqueta)[["periodo", "modelo", "segmento", "wape"]].to_dict(
        "records"
    )
por_periodo = pd.DataFrame(filas).pivot_table(
    index=["modelo", "periodo"], columns="segmento", values="wape", sort=False
)
display(por_periodo.loc[ORDEN].round(2))

## 3. Error mes a mes

In [ ]:
ref["ape"] = (ref["pred"] - ref["y"]).abs() / ref["y"] * 100
fig, ejes = plt.subplots(len(SEGMENTOS), 1, figsize=(10, 2.4 * len(SEGMENTOS)), sharex=True)
for eje, segmento in zip(ejes, SEGMENTOS, strict=True):
    g = ref[ref["segmento"] == segmento]
    for nombre, color in (
        ("ingenuo", exp_a.TENUE),
        ("estacional_ingenuo", exp_a.NARANJA),
        ("estacional_dias", exp_a.AZUL),
    ):
        linea = g[g["modelo"] == nombre]
        eje.plot(linea["mes"], linea["ape"], color=color, label=nombre.replace("_", " "))
    eje.set_title(f"{segmento} · error absoluto por mes (%)")
    eje.set_ylim(bottom=0)
    eje.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ejes[0].legend(loc="best", ncols=3)
plt.tight_layout()
plt.show()

## 4. ¿En qué meses del año falla el estacional ingenuo?

Error absoluto medio (%) por mes del año. Con menos de dos años de pruebas, cada celda sale de una o dos observaciones: es una pista, no una medida.

In [ ]:
estacional = ref[ref["modelo"] == "estacional_ingenuo"]
display(
    estacional.assign(mes_del_año=estacional["mes"].dt.month)
    .pivot_table(index="mes_del_año", columns="segmento", values="ape")
    .round(1)
)

## Decisiones

Las cifras quedan en las salidas de este notebook, que no se suben al repositorio.

1. **La vara oficial sigue siendo el estacional ingenuo**, porque es el método actual de Lojagas y el criterio de éxito de la tesis.
2. **El estacional ajustado por días de venta es una segunda vara, más exigente.** Mejora al estacional ingenuo sin entrenar nada: solo corrige la mezcla de días de la semana del mes. Un modelo que supere al estacional ingenuo pero no a este no aporta más que el calendario.
3. **El estacional ingenuo predice de menos de forma sistemática** en los segmentos que crecen: no conoce la tendencia. Es el margen más claro para un modelo.
4. **El ingenuo (último mes cerrado) no sirve como referencia en el agroindustrial**, por su ciclo semestral, y la media de 12 meses tampoco. En el industrial, en cambio, las cuatro referencias quedan casi iguales: no tiene estacionalidad que aprovechar.
5. **El error depende del período de prueba.** Toda comparación entre modelos se hace sobre los mismos meses.